# HYDRA-Net Stage 2: Train Cross-Modal Transformer on Anti-UAV

**Notebook purpose:** Train the Stage 2 cross-modal attention transformer on the Anti-UAV benchmark (RGB + IR frames).

**Runtime:** GPU required. Tested on Colab T4 and Colab Pro A100.

**Dataset:** Anti-UAV (CVPR 2021/2023 Challenge) — RGB and thermal-IR synchronized video of drones. [Challenge site](https://anti-uav.github.io/).

**Expected training time:** ~6 h on T4, ~2 h on A100.


## 1. Setup + GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU in Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/YOUR-USERNAME/hydra-net.git
%cd hydra-net
!pip install -q -r requirements.txt

## 2. Download Anti-UAV dataset

See `docs/DATASETS.md` for the registration and download procedure. The dataset is ~80 GB — cache it to Drive.


In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/hydra-net-data/antiuav')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Follow docs/DATASETS.md to download; structure should be:
#   antiuav/
#     train/
#       <sequence_id>/
#         visible.mp4
#         infrared.mp4
#         IR_label.json
#         visible_label.json
#     test/...

## 3. Dataset class

We build a PyTorch Dataset that emits synchronized (RGB frame, IR frame, audio-if-available, label) tuples. Anti-UAV has no audio — we either drop the audio token or pad with learned empty tokens (this is exactly what modality gating handles).


In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2
import json
import numpy as np


class AntiUAVDataset(Dataset):
    def __init__(self, root, split='train', image_size=224):
        self.root = Path(root) / split
        self.sequences = sorted([p for p in self.root.iterdir() if p.is_dir()])
        self.image_size = image_size
        self.frame_index = []  # (seq, frame_idx, label)
        self._build_index()
        self.rgb_tf = T.Compose([
            T.ToTensor(),
            T.Resize((image_size, image_size)),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.ir_tf = T.Compose([
            T.ToTensor(),
            T.Resize((image_size, image_size)),
            T.Normalize(mean=[0.5], std=[0.5]),
        ])

    def _build_index(self):
        for seq in self.sequences:
            label_file = seq / 'IR_label.json'
            if not label_file.exists():
                continue
            with open(label_file) as f:
                labels = json.load(f)
            # labels['exist'][i] is 1 if drone present in frame i
            for i, present in enumerate(labels['exist']):
                self.frame_index.append((seq, i, int(present)))

    def __len__(self):
        return len(self.frame_index)

    def _read_frame(self, video_path, frame_idx):
        cap = cv2.VideoCapture(str(video_path))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        cap.release()
        if not ok:
            return None
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        seq, frame_idx, label = self.frame_index[idx]
        rgb = self._read_frame(seq / 'visible.mp4', frame_idx)
        ir = self._read_frame(seq / 'infrared.mp4', frame_idx)
        if ir is not None:
            ir = cv2.cvtColor(ir, cv2.COLOR_RGB2GRAY)[..., None]
        rgb_t = self.rgb_tf(rgb) if rgb is not None else torch.zeros(3, self.image_size, self.image_size)
        ir_t = self.ir_tf(ir) if ir is not None else torch.zeros(1, self.image_size, self.image_size)
        return {'rgb': rgb_t, 'ir': ir_t, 'label': label}

## 4. Model, loss, training loop

We train the `CrossModalTransformer` with **modality dropout** during training so the model is robust to missing sensors at inference.


In [ ]:
from hydra_net.stage2 import CrossModalTransformer
import torch.nn.functional as F
import torch.optim as optim

device = 'cuda'
model = CrossModalTransformer(
    embed_dim=192,
    n_heads=6,
    n_layers=6,
    n_classes=2,  # binary drone/no-drone; swap to 5 for typed classification
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)


def modality_dropout(batch, drop_prob=0.2):
    """Randomly null out modalities during training."""
    if torch.rand(1).item() < drop_prob:
        batch['rgb'] = None
    if torch.rand(1).item() < drop_prob:
        batch['ir'] = None
    # Always keep at least one modality
    if batch['rgb'] is None and batch['ir'] is None:
        batch['rgb' if torch.rand(1).item() < 0.5 else 'ir'] = batch.get('rgb_orig') or batch.get('ir_orig')
    return batch


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_acc, n = 0, 0, 0
    for batch in loader:
        rgb = batch['rgb'].to(device)
        ir = batch['ir'].to(device)
        labels = batch['label'].to(device)

        # Stochastic modality dropout
        use_rgb = torch.rand(1).item() > 0.2
        use_ir = torch.rand(1).item() > 0.2 or not use_rgb

        out = model(
            rgb=rgb if use_rgb else None,
            ir=ir if use_ir else None,
        )
        loss = F.cross_entropy(out['logits'], labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_acc += (out['logits'].argmax(1) == labels).sum().item()
        n += labels.size(0)
    return total_loss / n, total_acc / n


# Full training run:
# N_EPOCHS = 10
# for epoch in range(N_EPOCHS):
#     loss, acc = train_epoch(model, train_loader, optimizer, device)
#     print(f'Epoch {epoch}: loss={loss:.4f}, acc={acc:.4f}')
#     scheduler.step()

## 5. Save model and measure GPU latency


In [ ]:
# Save
torch.save(model.state_dict(), '/content/drive/MyDrive/hydra-net-data/models/stage2_antiuav.pt')

# Latency
import time
model.eval()
rgb = torch.randn(1, 3, 224, 224).to(device)
ir = torch.randn(1, 1, 224, 224).to(device)

# Warmup
with torch.no_grad():
    for _ in range(20):
        _ = model(rgb=rgb, ir=ir)

# Measure
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(100):
        _ = model(rgb=rgb, ir=ir)
torch.cuda.synchronize()
elapsed = (time.perf_counter() - t0) / 100 * 1000
print(f'Stage 2 GPU latency: {elapsed:.2f} ms per sample')